In [1]:
import torch
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained('Peltarion/dnabert-distilbert')


/home/jbs1009/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jbs1009/.local/lib/python3.8/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/home/jbs1009/.local/lib/python3.8/site-packages/transformers/modeling_utils.py:446: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, th

In [2]:
import numpy as np
import pandas as pd
import torch
import gc
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, classification_report
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback

2026-06-03 18:43:50.356720: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-03 18:43:50.392740: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-03 18:43:50.989547: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [5]:
def kmeriza(secuencia, k = 6):
    return " ".join([secuencia[i:i+k] for i in range(len(secuencia) - k + 1)])


base = pd.read_csv("/home/jbs1009/TFM/datosGene4PD/base_aumentada_flancos50_6030.csv")
etiqueta_binaria = {"Sano": 0, "Riesgo_PD": 1}
base["labels"] = base["labels"].map(etiqueta_binaria)

base["Kmers"] = base["Secuencia"].apply(lambda x: kmeriza(x.lower()))

gss1 = GroupShuffleSplit(n_splits = 1, test_size = 0.2, random_state = 2026)
id_tv, id_test = next(gss1.split(base, base["labels"], groups = base["rsID"]))

base_tv = base.iloc[id_tv].reset_index(drop = True)
base_test = base.iloc[id_test].reset_index(drop = True)

gss2 = GroupShuffleSplit(n_splits = 1, test_size = 0.25, random_state = 2026)
id_train, id_val = next(gss2.split(base_tv, base_tv["labels"], groups = base_tv["rsID"]))

base_train = base_tv.iloc[id_train].reset_index(drop = True)
base_val = base_tv.iloc[id_val].reset_index(drop = True)

hf_dataset = DatasetDict({
    "train": Dataset.from_pandas(base_train),
    "validation": Dataset.from_pandas(base_val),
    "test": Dataset.from_pandas(base_test)
})


tokenizer = AutoTokenizer.from_pretrained('armheb/DNA_bert_6', trust_remote_code = True)

def tokeniza_secuencias(batch):
    return tokenizer(batch["Kmers"], truncation = True, max_length = 128)

hf_base_tokenizada = hf_dataset.map(
    tokeniza_secuencias,
    batched = True,
    batch_size = 10
)
gc.collect()


def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        logits = predictions[0]
    
    else:
        logits = predictions

    probs = torch.nn.functional.softmax(torch.tensor(logits), dim = -1).numpy()[:, 1]
    preds = np.argmax(logits, axis = 1)

    return {
        "roc_auc": roc_auc_score(labels, probs),
        "pr_auc": average_precision_score(labels, probs),
        "f1": f1_score(labels, preds)
    }


model = AutoModelForSequenceClassification.from_pretrained(
    'Peltarion/dnabert-distilbert', 
    num_labels = 2
)

data_collator = DataCollatorWithPadding(tokenizer = tokenizer)

training_args = TrainingArguments(
    output_dir="./resultados_distilbert_kmers",  
    learning_rate=2e-5,                 
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,      
    per_device_eval_batch_size=16,
    num_train_epochs=15,                 
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    evaluation_strategy="epoch",             
    save_strategy="epoch",
    fp16=True,                   
    dataloader_num_workers=0,
    load_best_model_at_end=True,
    metric_for_best_model="roc_auc",
    greater_is_better=True    
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_base_tokenizada["train"],
    eval_dataset=hf_base_tokenizada["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=15)]
)


trainer.train()



print("EVALUACIÓN EN CONJUNTO DE TEST")

predicciones = trainer.predict(hf_base_tokenizada["test"])

if isinstance(predicciones.predictions, tuple):
    logits_test = predicciones.predictions[0]

else:
    logits_test = predicciones.predictions

labels_test = predicciones.label_ids

probs_test = torch.nn.functional.softmax(torch.tensor(logits_test), dim = 1).numpy()[:, 1]
preds_test = np.argmax(logits_test, axis = 1)

print(f"ROC-AUC Test: {roc_auc_score(labels_test, probs_test):.4f}")
print(f"PR-AUC Test: {average_precision_score(labels_test, probs_test):.4f}\n")

print(classification_report(labels_test, preds_test, target_names = ["Sano", "Riesgo_PD"]))

/home/jbs1009/.local/lib/python3.8/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
A new version of the following files was downloaded from https://huggingface.co/armheb/DNA_bert_6:
- configuration_bert.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
Map: 100%|██████████| 1178/1178 [00:00<00:00, 5572.30 examples/s]
/home/jbs1009/.local/lib/python3.8/site-packages/transformers/modeling_utils.py:446: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pyto

Epoch,Training Loss,Validation Loss,Roc Auc,Pr Auc,F1
0,No log,0.699443,0.500000,0.496867,0.000000
2,No log,0.697043,0.500000,0.496867,0.663876
2,0.703100,0.699646,0.500000,0.496867,0.663876
4,0.703100,0.717164,0.500000,0.496867,0.663876
4,0.700300,0.697310,0.500000,0.496867,0.663876
6,0.700300,0.693497,0.500000,0.496867,0.663876
6,0.696900,0.693234,0.500000,0.496867,0.000000
8,0.696900,0.695732,0.500000,0.496867,0.663876
8,0.696300,0.693816,0.500000,0.496867,0.663876
10,0.696300,0.693761,0.500000,0.496867,0.663876


/home/jbs1009/.local/lib/python3.8/site-packages/transformers/trainer.py:2700: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  else torch.cuda.amp.autocast(cache_enabled=cache_enabled, dtype=self.amp_dtype)
/home/jbs1009/.local/lib/python3.8/site-packages/transformers/trainer.py:2700: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  else torch.cuda.amp.autocast(cache_enabled=cache_enabled, dtype=self.amp_dtype)
/home/jbs1009/.local/lib/python3.8/site-packages/transformers/trainer.py:2700: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  else torch.cuda.amp.autocast(cache_enabled=cache_enabled, dtype=self.amp_dtype)
/home/jbs1009/.local/lib/python3.8/site-packages/transformers/trainer.py:2700: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Plea

EVALUACIÓN EN CONJUNTO DE TEST


/home/jbs1009/.local/lib/python3.8/site-packages/transformers/trainer.py:2700: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  else torch.cuda.amp.autocast(cache_enabled=cache_enabled, dtype=self.amp_dtype)


ROC-AUC Test: 0.5000
PR-AUC Test: 0.4881

              precision    recall  f1-score   support

        Sano       0.51      1.00      0.68       603
   Riesgo_PD       0.00      0.00      0.00       575

    accuracy                           0.51      1178
   macro avg       0.26      0.50      0.34      1178
weighted avg       0.26      0.51      0.35      1178



/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/jbs1009/.local/lib/python3.8/site-packages/sklearn/metrics/_classification.py:1471: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
